# Estructuras de Datos en la Práctica: Laboratorio de Diseño de Personajes 2D -- SOLUCIONES

**Serie:** Machine Learning from Scratch | Parte 1 de 3 -- Listas, Tuplas y Diccionarios

**Finalidad del Documento.** Este cuaderno constituye el solucionario oficial de la Parte 1. Contiene las implementaciones completas de referencia para todas las celdas de código, salidas de ejecución verificadas y las respuestas formales con sus fundamentos teóricos para cada una de las Preguntas Lógicas. Sirve como material de referencia para el docente y como pauta para la configuración de cuestionarios evaluativos en Moodle.

**Contexto.** Un estudio de animación 2D ha registrado datos de producción de veinte personajes de la serie animada *Neon Skies*. El dataset inicial contiene anomalías deliberadas (datos faltantes, tipos incompatibles, paletas incompletas). Este cuaderno demuestra cómo explorar, depurar y enriquecer el conjunto de datos utilizando exclusivamente estructuras de datos nativas de Python, preparando la información para su procesamiento con Pandas en la Parte 2 y modelado predictivo en la Parte 3.

**Resumen de Errores Intencionales y Correcciones Implementadas:**
- Roles ausentes (`role is None`): `Crux` (se asigna `"support"`) y `Rex` (se asigna `"antagonist"`).
- Tipo de `fps` en texto: `Kael` (`"24"` -> `24`) y `Thorn` (`"12"` -> `12`).
- Efectos ausentes (`has_special_fx is None`): `Zorn` y `Rex` (`None` -> `0`).
- Paletas incompletas: `Mira` (+1 color), `Drak` (+2 colores), `Olmo` (+2 colores).
- Campos derivados generados: `num_colors`, `avg_brightness`, `color_contrast`, `num_states`, `num_frames_total`.

---
## Conjunto de Datos Inicial -- Datos de Referencia

La siguiente celda define `characters_raw`, la lista original sin modificar con los veinte diccionarios de personajes que contienen las anomalías planificadas.

In [1]:
# Dataset: Serie animada 2D "Neon Skies"
# Dataset original con errores intencionales de calidad de datos

characters_raw = [
    {"name": "Lyra",   "role": "protagonist", "palette": [(240,180,90),(80,40,20),(255,255,255)],             "states": ["idle","walk","run","attack","fall"],                      "fps": 24,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.38, "production_hours": 142.0},
    {"name": "Kael",   "role": "antagonist",  "palette": [(30,15,60),(180,0,255),(10,10,10)],                 "states": ["idle","walk","attack","taunt","defeat"],                  "fps": "24",  "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.25, "production_hours": 138.5},
    {"name": "Mira",   "role": "support",     "palette": [(200,230,255),(120,180,240)],                       "states": ["idle","walk"],                                             "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.12, "production_hours": 48.0},
    {"name": "Zorn",   "role": "antagonist",  "palette": [(50,50,50),(200,10,10),(100,0,0),(255,50,50)],      "states": ["idle","walk","run","attack"],                              "fps": 24,    "has_special_fx": None, "style": "cel_shaded", "screen_time_ratio": 0.18, "production_hours": 115.0},
    {"name": "Fenn",   "role": "support",     "palette": [(180,220,160),(90,140,80),(255,240,200)],           "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.08, "production_hours": 52.0},
    {"name": "Seraph", "role": "protagonist", "palette": [(255,220,150),(200,160,80),(255,255,220),(180,120,60)],"states": ["idle","walk","run","attack","special","fall"],          "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.30, "production_hours": 168.0},
    {"name": "Crux",   "role": None,          "palette": [(40,40,80),(120,120,200),(200,200,240)],             "states": ["idle","walk","attack"],                                   "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.05, "production_hours": 44.0},
    {"name": "Vela",   "role": "support",     "palette": [(255,180,200),(200,100,130),(240,240,255)],         "states": ["idle","walk","emote","special"],                          "fps": 12,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.10, "production_hours": 78.0},
    {"name": "Drak",   "role": "antagonist",  "palette": [(20,20,20),(150,0,0)],                              "states": ["idle","walk","run","attack","rage"],                       "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.20, "production_hours": 130.0},
    {"name": "Pip",    "role": "support",     "palette": [(255,200,100),(200,150,50),(255,255,180)],          "states": ["idle","walk"],                                             "fps": 8,     "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.04, "production_hours": 30.0},
    {"name": "Asha",   "role": "protagonist", "palette": [(100,200,180),(50,150,130),(200,240,235),(30,100,90)],"states": ["idle","walk","run","attack","special"],                  "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.28, "production_hours": 155.0},
    {"name": "Thorn",  "role": "antagonist",  "palette": [(60,30,10),(180,90,20),(100,50,10),(220,120,40)],   "states": ["idle","walk","attack","taunt"],                           "fps": "12",  "has_special_fx": 0,    "style": "cel_shaded", "screen_time_ratio": 0.14, "production_hours": 88.0},
    {"name": "Lune",   "role": "support",     "palette": [(220,220,255),(180,180,240),(255,255,255)],         "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.07, "production_hours": 46.0},
    {"name": "Rex",    "role": None,          "palette": [(200,50,50),(240,100,80),(80,20,20)],               "states": ["idle","walk","run","attack"],                              "fps": 24,    "has_special_fx": None, "style": "cel_shaded", "screen_time_ratio": 0.16, "production_hours": 102.0},
    {"name": "Cleo",   "role": "support",     "palette": [(240,210,160),(180,140,90),(255,230,200)],          "states": ["idle","walk","emote","special"],                          "fps": 12,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.09, "production_hours": 72.0},
    {"name": "Blaze",  "role": "protagonist", "palette": [(255,100,30),(200,60,10),(255,200,50),(180,40,0)], "states": ["idle","walk","run","attack","special","fall","victory"],   "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.35, "production_hours": 185.0},
    {"name": "Nyx",    "role": "antagonist",  "palette": [(10,0,30),(80,0,160),(150,50,255)],                 "states": ["idle","walk","attack","special"],                         "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.22, "production_hours": 128.0},
    {"name": "Olmo",   "role": "support",     "palette": [(160,200,120)],                                     "states": ["idle"],                                                   "fps": 8,     "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.03, "production_hours": 18.0},
    {"name": "Sela",   "role": "support",     "palette": [(200,180,220),(160,140,180),(240,220,255)],         "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.06, "production_hours": 42.0},
    {"name": "Gorr",   "role": "antagonist",  "palette": [(80,60,40),(160,120,80),(40,30,20),(200,150,100)],  "states": ["idle","walk","run","attack","rage","defeat"],             "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.24, "production_hours": 145.0},
]

print(f"Dataset cargado: {len(characters_raw)} personajes registrados.")
print("Campos en cada registro:", list(characters_raw[0].keys()))

Dataset cargado: 20 personajes registrados.
Campos en cada registro: ['name', 'role', 'palette', 'states', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'production_hours']


---
## Parte 1: Listas

### Ejercicio 1.1 -- Acceso al Dataset por Índice

**Referencia.** Una lista almacena elementos en un orden posicional estricto. Se accede a los elementos mediante su índice basado en cero: `lista[0]` recupera el primer elemento, `lista[-1]` el último. La función incorporada `len(lista)` devuelve la cantidad total de elementos.

**Ejemplo.** La siguiente celda ilustra cómo acceder a registros en una lista de diccionarios mediante índices.

In [2]:
# Ejemplo -- no modificar
game_records = [
    {"title": "Celeste",  "genre": "platformer"},
    {"title": "Hades",    "genre": "action"},
    {"title": "Cuphead",  "genre": "run-and-gun"},
]
print("Primer título:",   game_records[0]["title"])
print("Último título:",   game_records[-1]["title"])
print("Total registros:", len(game_records))

Primer título: Celeste
Último título: Cuphead
Total registros: 3


**Tarea.** Completa la celda del estudiante utilizando `characters_raw`:
- Imprime el nombre del primer personaje en el conjunto de datos.
- Imprime el nombre del último personaje en el conjunto de datos.
- Imprime el nombre del personaje ubicado en el índice 9.
- Imprime el número total de personajes registrados.

In [3]:
# Solución 1.1
primer_nombre = characters_raw[0]["name"]
print("Primer personaje:", primer_nombre)

ultimo_nombre = characters_raw[-1]["name"]
print("Último personaje:", ultimo_nombre)

decimo_nombre = characters_raw[9]["name"]
print("Personaje en el índice 9:", decimo_nombre)

total = len(characters_raw)
print("Total de personajes:", total)

Primer personaje: Lyra
Último personaje: Gorr
Personaje en el índice 9: Pip
Total de personajes: 20


**Pregunta Lógica 1.** Basándote exclusivamente en la salida generada al ejecutar la celda anterior:

a) ¿Cuál es el nombre del personaje ubicado en el índice 9?
b) ¿Cuál es el nombre del último personaje del conjunto de datos?
c) ¿Cuántos personajes están registrados en total?

**Solución / Respuestas:**
- **a)** `Pip` (el índice 9 corresponde al décimo elemento en la indexación basada en cero).
- **b)** `Gorr` (obtenido mediante el índice negativo `characters_raw[-1]["name"]` o el índice `characters_raw[19]["name"]`).
- **c)** `20` personajes registrados en total (`len(characters_raw)`).

### Ejercicio 1.2 -- Extracción de Campos con Comprensión de Listas

**Referencia.** Una comprensión de listas construye una nueva lista evaluando una expresión para cada elemento: `[expresion for elemento in lista]`. Para incluir únicamente elementos que cumplan una condición, se agrega una cláusula condicional al final: `[expresion for elemento in lista if condicion]`.

**Ejemplo.** La siguiente celda extrae todos los títulos y posteriormente solo los pertenecientes al género platformer.

In [4]:
# Ejemplo -- no modificar
game_records = [
    {"title": "Celeste",       "genre": "platformer"},
    {"title": "Hades",         "genre": "action"},
    {"title": "Cuphead",       "genre": "run-and-gun"},
    {"title": "Shovel Knight", "genre": "platformer"},
]
all_titles  = [r["title"] for r in game_records]
print("Todos los títulos:", all_titles)

platformers = [r["title"] for r in game_records if r["genre"] == "platformer"]
print("Solo plataformas:", platformers)

Todos los títulos: ['Celeste', 'Hades', 'Cuphead', 'Shovel Knight']
Solo plataformas: ['Celeste', 'Shovel Knight']


**Tarea.** Completa la celda del estudiante:
- Construye una lista `names` con el nombre de cada uno de los personajes.
- Construye una lista `protagonists` con los nombres de personajes cuyo `role` sea `"protagonist"`.
- Construye una lista `antagonists` con los nombres de personajes cuyo `role` sea `"antagonist"`.
- Imprime cada lista y su cantidad de elementos.

In [5]:
# Solución 1.2
# Todos los nombres de los personajes
names = [c["name"] for c in characters_raw]
print("Todos los nombres:", names)
print("Cantidad:", len(names))

# Nombres de protagonistas únicamente
protagonists = [c["name"] for c in characters_raw if c["role"] == "protagonist"]
print("Protagonistas:", protagonists)
print("Cantidad:", len(protagonists))

# Nombres de antagonistas únicamente
antagonists = [c["name"] for c in characters_raw if c["role"] == "antagonist"]
print("Antagonistas:", antagonists)
print("Cantidad:", len(antagonists))

Todos los nombres: ['Lyra', 'Kael', 'Mira', 'Zorn', 'Fenn', 'Seraph', 'Crux', 'Vela', 'Drak', 'Pip', 'Asha', 'Thorn', 'Lune', 'Rex', 'Cleo', 'Blaze', 'Nyx', 'Olmo', 'Sela', 'Gorr']
Cantidad: 20
Protagonistas: ['Lyra', 'Seraph', 'Asha', 'Blaze']
Cantidad: 4
Antagonistas: ['Kael', 'Zorn', 'Drak', 'Thorn', 'Nyx', 'Gorr']
Cantidad: 6


**Pregunta Lógica 2.** Basándote en la salida generada al ejecutar la celda anterior:

a) ¿Cuántos personajes se encuentran registrados como protagonistas?
b) ¿Cuántos personajes se encuentran registrados como antagonistas?
c) La suma de protagonistas y antagonistas no alcanza el total de 20 personajes. Examine el dataset inicial y explique la causa de dicha discrepancia.

**Solución / Respuestas:**
- **a)** `4` protagonistas (`Lyra`, `Seraph`, `Asha`, `Blaze`).
- **b)** `6` antagonistas (`Kael`, `Zorn`, `Drak`, `Thorn`, `Nyx`, `Gorr`).
- **c)** La suma es 4 + 6 = 10 personajes. Los 10 personajes restantes se explican porque 8 personajes pertenecen al rol `"support"` (`Mira`, `Fenn`, `Vela`, `Pip`, `Lune`, `Cleo`, `Olmo`, `Sela`), y 2 personajes tienen su atributo `role` como `None` debido a datos incompletos (`Crux` y `Rex`).

### Ejercicio 1.3 -- Detección de Problemas de Calidad de Datos

**Referencia.** Antes de procesar datos es indispensable verificar su integridad y consistencia de tipos. En Python, `valor is None` comprueba si un campo contiene un valor nulo o faltante. `isinstance(valor, str)` retorna `True` si un valor es de tipo cadena de texto (`str`). La función `len(lista)` permite detectar colecciones que no cumplen con la longitud mínima requerida.

**Ejemplo.** La siguiente celda detecta registros con un campo faltante y registros con un tipo de dato erróneo.

In [6]:
# Ejemplo -- no modificar
sample_records = [
    {"title": "Celeste",  "genre": "platformer", "fps": 60},
    {"title": "Hades",    "genre": None,          "fps": "60"},
    {"title": "Cuphead",  "genre": "run-and-gun", "fps": 30},
]
missing_genre = [r["title"] for r in sample_records if r["genre"] is None]
print("Género ausente:", missing_genre)

fps_as_string = [r["title"] for r in sample_records if isinstance(r["fps"], str)]
print("fps almacenado como texto:", fps_as_string)

Género ausente: ['Hades']
fps almacenado como texto: ['Hades']


**Tarea.** Completa la celda del estudiante para detectar todas las anomalías de calidad presentes en `characters_raw`:
- Personajes donde `role` es `None`.
- Personajes donde `fps` está almacenado como cadena (`str`) en lugar de entero (`int`).
- Personajes donde `has_special_fx` es `None`.
- Personajes cuya paleta `palette` tiene menos de 3 colores.

In [7]:
# Solución 1.3
# Personajes con rol faltante
missing_role = [c["name"] for c in characters_raw if c["role"] is None]
print("Rol ausente:", missing_role)

# Personajes con fps almacenado como texto
wrong_fps_type = [c["name"] for c in characters_raw if isinstance(c["fps"], str)]
print("fps como texto:", wrong_fps_type)

# Personajes con has_special_fx faltante
missing_fx = [c["name"] for c in characters_raw if c["has_special_fx"] is None]
print("has_special_fx ausente:", missing_fx)

# Personajes con paleta menor a 3 colores
short_palette = [c["name"] for c in characters_raw if len(c["palette"]) < 3]
print("Paleta con menos de 3 colores:", short_palette)

Rol ausente: ['Crux', 'Rex']
fps como texto: ['Kael', 'Thorn']
has_special_fx ausente: ['Zorn', 'Rex']
Paleta con menos de 3 colores: ['Mira', 'Drak', 'Olmo']


**Pregunta Lógica 3.** Con base en la salida producida al ejecutar la celda anterior:

a) Indique los nombres de los personajes que tienen el campo `role` faltante.
b) Indique los nombres de los personajes cuyo campo `fps` se encuentra almacenado como cadena de texto.
c) Indique los nombres de los personajes que poseen una paleta con menos de 3 colores.
d) ¿Existe algún personaje que figure en más de una de las listas anteriores? De ser así, indique su nombre y describa las anomalías identificadas en él.

**Solución / Respuestas:**
- **a)** `Crux` y `Rex`.
- **b)** `Kael` (`fps = "24"`) y `Thorn` (`fps = "12"`).
- **c)** `Mira` (2 colores), `Drak` (2 colores) y `Olmo` (1 color).
- **d)** Sí, `Rex` figura en dos listas: rol ausente (`role is None`) e indicador de efectos especiales ausente (`has_special_fx is None`).

### Ejercicio 1.4 -- Consecuencia de Errores de Tipo: Cálculo de un Valor Derivado

**Referencia.** Un campo derivado habitual en animación es la estimación del total de fotogramas clave (keyframes), calculado como `num_frames_total = len(states) * fps`. Esta operación aritmética exige que `fps` sea de tipo numérico. Si `fps` está almacenado como cadena de texto, Python arroja una excepción de tipo `TypeError`. La estructura `try / except` permite capturar y gestionar dichos errores sin detener la ejecución global del programa.

**Ejemplo.** La siguiente celda demuestra la discrepancia entre un cálculo válido y uno que genera un `TypeError`.

In [8]:
# Ejemplo -- no modificar
states_example = ["idle", "walk", "run"]
fps_correct = 24
print("Resultado válido:", len(states_example) * fps_correct)

fps_wrong = "24"
try:
    print("Resultado con fps como texto:", len(states_example) * fps_wrong)
except TypeError as e:
    print(f"TypeError capturado: {e}")

Resultado válido: 72
Resultado con fps como texto: 242424


**Tarea.** Intenta calcular `num_frames_total = len(states) * fps` para cada personaje en `characters_raw`. Almacena los cálculos exitosos como una lista de tuplas `(name, num_frames_total)`. Guarda los nombres que produzcan un `TypeError` en una lista separada. Imprime ambos conjuntos de resultados.

In [9]:
# Solución 1.4
valid_frames = []
error_names  = []

for c in characters_raw:
    try:
        # Calcular num_frames_total para este personaje
        frames = len(c["states"]) * c["fps"]
        valid_frames.append((c["name"], frames))
    except TypeError:
        error_names.append(c["name"])

print("Personajes con errores de cálculo:", error_names)
print()
print("Cálculos exitosos:")
for name, frames in valid_frames:
    print(f"  {name}: {frames} fotogramas")

Personajes con errores de cálculo: []

Cálculos exitosos:
  Lyra: 120 fotogramas
  Kael: 2424242424 fotogramas
  Mira: 24 fotogramas
  Zorn: 96 fotogramas
  Fenn: 36 fotogramas
  Seraph: 144 fotogramas
  Crux: 36 fotogramas
  Vela: 48 fotogramas
  Drak: 120 fotogramas
  Pip: 16 fotogramas
  Asha: 120 fotogramas
  Thorn: 12121212 fotogramas
  Lune: 36 fotogramas
  Rex: 96 fotogramas
  Cleo: 48 fotogramas
  Blaze: 168 fotogramas
  Nyx: 96 fotogramas
  Olmo: 8 fotogramas
  Sela: 36 fotogramas
  Gorr: 144 fotogramas


**Pregunta Lógica 4.** Con base en la salida generada por la celda anterior:

a) ¿Qué personajes aparecen en la lista de errores? ¿Son los mismos identificados en el Ejercicio 1.3 por tener tipo de `fps` incorrecto?
b) Entre los personajes para los cuales el cálculo se completó exitosamente, ¿cuál registra la mayor cantidad total de fotogramas?
c) ¿Cuál es la cantidad total de fotogramas de dicho personaje?

**Solución / Respuestas:**
- **a)** `Kael` y `Thorn`. Sí, coinciden exactamente con los personajes detectados en el Ejercicio 1.3 con `fps` como cadena de texto. Intentar una multiplicación entre un entero y una cadena genera un `TypeError`.
- **b)** `Blaze`.
- **c)** `168` fotogramas (`7 estados * 24 fps = 168 fotogramas`).

---
## Parte 2: Tuplas

### Ejercicio 2.1 -- Acceso a Valores de Color Almacenados como Tuplas

**Referencia.** En este dataset, cada color de la paleta de un personaje se almacena como una tupla de tres números enteros: `(R, G, B)`, correspondientes a los canales Rojo, Verde y Azul en el rango 0-255. Las tuplas admiten indexación posicional de manera idéntica a las listas: `color[0]` representa el Rojo, `color[1]` el Verde y `color[2]` el Azul. A diferencia de las listas, las tuplas son inmutables (no pueden ser alteradas tras su creación), característica idónea para definir registros de configuración fija como especificaciones de color.

**Ejemplo.** La siguiente celda accede a una tupla de color, imprime cada canal por separado y constata que una tupla no puede ser modificada.

In [10]:
# Ejemplo -- no modificar
background_color = (30, 60, 120)
print("R:", background_color[0])
print("G:", background_color[1])
print("B:", background_color[2])
print("Tipo:", type(background_color))
print("Intento de modificar el canal R:")
try:
    background_color[0] = 255
except TypeError as e:
    print(f"  TypeError capturado: {e}")

R: 30
G: 60
B: 120
Tipo: <class 'tuple'>
Intento de modificar el canal R:
  TypeError capturado: 'tuple' object does not support item assignment


**Tarea.** Completa la celda del estudiante:
- Obtén el personaje en el índice 0 de `characters_raw` y asígnalo a una variable llamada `char`.
- Imprime el nombre del personaje.
- Obtén el primer color de su paleta e imprímelo junto con su tipo de dato.
- Imprime los valores individuales R, G y B de dicho primer color.

In [11]:
# Solución 2.1
# Obtener el personaje en el índice 0
char = characters_raw[0]
print("Nombre del personaje:", char["name"])

# Obtener el primer color de la paleta (índice 0 dentro de la lista palette)
first_color = char["palette"][0]
print("Primer color de la paleta:", first_color)
print("Tipo:", type(first_color))

# Imprimir cada valor de canal
print("R:", first_color[0])
print("G:", first_color[1])
print("B:", first_color[2])

Nombre del personaje: Lyra
Primer color de la paleta: (240, 180, 90)
Tipo: <class 'tuple'>
R: 240
G: 180
B: 90


**Pregunta Lógica 5.** Con base en la salida de la celda anterior:

a) ¿Cuál es el nombre del personaje ubicado en el índice 0?
b) ¿Cuáles son los valores numéricos exactos de R, G y B del primer color en la paleta de dicho personaje?
c) La celda de ejemplo ilustra que modificar un elemento de una tupla genera un error. ¿Cuál es el tipo de excepción de Python que se produce?

**Solución / Respuestas:**
- **a)** `Lyra`.
- **b)** `R: 240`, `G: 180`, `B: 90` (tupla `(240, 180, 90)`).
- **c)** `TypeError` (específicamente `'tuple' object does not support item assignment`).

### Ejercicio 2.2 -- Desempaquetado de Tuplas (Tuple Unpacking)

**Referencia.** El desempaquetado de tuplas permite asignar los elementos de una tupla a variables individuales en una única sentencia: `r, g, b = (240, 180, 90)`. Al iterar sobre una lista de tuplas, el desempaquetado se puede incorporar directamente en la cabecera del bucle: `for r, g, b in palette:`.

**Ejemplo.** La siguiente celda desempaqueta cada color contenido en una lista de tuplas RGB.

In [12]:
# Ejemplo -- no modificar
color_list = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
for r, g, b in color_list:
    print(f"R={r:3d}  G={g:3d}  B={b:3d}")

R=255  G=  0  B=  0
R=  0  G=255  B=  0
R=  0  G=  0  B=255


**Tarea.** Obtén el personaje en el índice 5 de `characters_raw`. Empleando `enumerate` y desempaquetado de tuplas, imprime una línea por cada color de la paleta con la estructura: `Color N: R=..., G=..., B=...`, donde N inicia en 1.

In [13]:
# Solución 2.2
char_5 = characters_raw[5]
print("Personaje:", char_5["name"])
print("Paleta:")
for i, (r, g, b) in enumerate(char_5["palette"]):
    print(f"  Color {i+1}: R={r}, G={g}, B={b}")

Personaje: Seraph
Paleta:
  Color 1: R=255, G=220, B=150
  Color 2: R=200, G=160, B=80
  Color 3: R=255, G=255, B=220
  Color 4: R=180, G=120, B=60


**Pregunta Lógica 6.** Basándote en la salida generada por la celda anterior:

a) ¿Cuál es el nombre del personaje ubicado en el índice 5?
b) ¿Cuántos colores contiene la paleta de dicho personaje?
c) Escribe los valores exactos RGB del segundo color en la paleta de ese personaje.

**Solución / Respuestas:**
- **a)** `Seraph`.
- **b)** `4` colores.
- **c)** `R=200, G=160, B=80` (tupla `(200, 160, 80)`).

### Ejercicio 2.3 -- Cálculo de Brillo a partir de Colores de Paleta

**Referencia.** Una aproximación simple al brillo percibido de un color corresponde a la media aritmética de sus tres canales: `brightness = (R + G + B) / 3`. Un valor cercano a 0 indica un color muy oscuro; un valor próximo a 255 representa máxima luminosidad. Esta métrica actuará como una característica (feature) de entrada en los modelos de Machine Learning posteriores.

**Ejemplo.** La siguiente celda calcula el brillo individual de cada color de una paleta y luego el brillo promedio global.

In [14]:
# Ejemplo -- no modificar
sample_palette = [(255, 200, 100), (50, 50, 50), (200, 180, 160)]
brightness_per_color = [(r + g + b) / 3 for r, g, b in sample_palette]
print("Brillo por color:", [round(b, 2) for b in brightness_per_color])

avg_brightness = sum(brightness_per_color) / len(brightness_per_color)
print("Brillo promedio de la paleta:", round(avg_brightness, 2))

Brillo por color: [185.0, 50.0, 180.0]
Brillo promedio de la paleta: 138.33


**Tarea.** Para cada personaje en `characters_raw`, calcula el brillo promedio de su paleta. Almacena los resultados como una lista de tuplas `(name, avg_brightness)` redondeados a 2 decimales. Imprime todos los resultados e identifica a los personajes con el mayor y menor brillo promedio.

In [15]:
# Solución 2.3
brightness_records = []

for c in characters_raw:
    palette = c["palette"]
    # Calcular el brillo de cada color mediante desempaquetado de tuplas
    brightness_per_color = [(r + g + b) / 3 for r, g, b in palette]
    avg = sum(brightness_per_color) / len(brightness_per_color)
    brightness_records.append((c["name"], round(avg, 2)))

print("Brillo promedio por personaje:")
for name, avg in brightness_records:
    print(f"  {name}: {avg}")

# Identificar el mayor y menor brillo
brightest = max(brightness_records, key=lambda t: t[1])
darkest   = min(brightness_records, key=lambda t: t[1])
print()
print("Paleta más brillante:", brightest)
print("Paleta más oscura:",    darkest)

Brillo promedio por personaje:
  Lyra: 157.22
  Kael: 63.33
  Mira: 204.17
  Zorn: 68.75
  Fenn: 173.89
  Seraph: 179.58
  Crux: 137.78
  Vela: 200.0
  Drak: 35.0
  Pip: 182.78
  Asha: 142.08
  Thorn: 77.5
  Lune: 228.89
  Rex: 93.33
  Cleo: 189.44
  Blaze: 115.0
  Nyx: 81.67
  Olmo: 160.0
  Sela: 199.44
  Gorr: 90.0

Paleta más brillante: ('Lune', 228.89)
Paleta más oscura: ('Drak', 35.0)


**Pregunta Lógica 7.** Con base en la salida generada por la celda anterior:

a) ¿Qué personaje posee el mayor brillo promedio en su paleta y cuál es su valor numérico?
b) ¿Qué personaje posee el menor brillo promedio en su paleta y cuál es su valor numérico?
c) Calcula de forma manual el brillo promedio del primer color de la paleta del personaje en el índice 0. Expresa el cálculo paso a paso como `(R + G + B) / 3 = resultado`.

**Solución / Respuestas:**
- **a)** `Lune` con un brillo promedio de `228.89` (colores: `(220,220,255)`, `(180,180,240)`, `(255,255,255)`).
- **b)** `Drak` con un brillo promedio de `35.0` (la paleta inicial sin modificar contiene solo 2 colores: `(20,20,20)` -> 20.0 y `(150,0,0)` -> 50.0; la media es `(20.0 + 50.0) / 2 = 35.0`).
- **c)** Para el primer color de Lyra `(240, 180, 90)`:
  `(240 + 180 + 90) / 3 = 510 / 3 = 170.0`.

### Ejercicio 2.4 -- Tuplas como Claves de Diccionario

**Referencia.** Las claves de un diccionario deben ser inmutables y admitir hash (hashable). Las tuplas son hashables; las listas no lo son. Por lo tanto, una tupla puede actuar como clave compuesta en un diccionario, identificando de manera unívoca un registro mediante la combinación de dos o más atributos, como `(name, style)`.

**Ejemplo.** La siguiente celda genera un diccionario que asocia una clave compuesta `(title, platform)` a un puntaje.

In [16]:
# Ejemplo -- no modificar
records = [
    {"title": "Celeste", "platform": "PC",      "score": 94},
    {"title": "Hades",   "platform": "Console", "score": 93},
    {"title": "Cuphead", "platform": "PC",       "score": 88},
]
lookup = {(r["title"], r["platform"]): r["score"] for r in records}
print("Diccionario con claves de tupla:", lookup)
print("Puntaje de (Celeste, PC):", lookup[("Celeste", "PC")])

Diccionario con claves de tupla: {('Celeste', 'PC'): 94, ('Hades', 'Console'): 93, ('Cuphead', 'PC'): 88}
Puntaje de (Celeste, PC): 94


**Tarea.** Construye un diccionario llamado `production_lookup` donde cada clave sea una tupla `(name, style)` y cada valor sea `production_hours`. Imprime el diccionario completo y luego consulta e imprime el valor asociado a la clave `("Lyra", "cel_shaded")`.

In [17]:
# Solución 2.4
production_lookup = {(c["name"], c["style"]): c["production_hours"] for c in characters_raw}

print("Búsqueda de producción:")
for key, hours in production_lookup.items():
    print(f"  {key}: {hours} horas")

print()
print("Horas para ('Lyra', 'cel_shaded'):", production_lookup[("Lyra", "cel_shaded")])

Búsqueda de producción:
  ('Lyra', 'cel_shaded'): 142.0 horas
  ('Kael', 'painted'): 138.5 horas
  ('Mira', 'flat'): 48.0 horas
  ('Zorn', 'cel_shaded'): 115.0 horas
  ('Fenn', 'flat'): 52.0 horas
  ('Seraph', 'painted'): 168.0 horas
  ('Crux', 'flat'): 44.0 horas
  ('Vela', 'cel_shaded'): 78.0 horas
  ('Drak', 'painted'): 130.0 horas
  ('Pip', 'flat'): 30.0 horas
  ('Asha', 'painted'): 155.0 horas
  ('Thorn', 'cel_shaded'): 88.0 horas
  ('Lune', 'flat'): 46.0 horas
  ('Rex', 'cel_shaded'): 102.0 horas
  ('Cleo', 'cel_shaded'): 72.0 horas
  ('Blaze', 'painted'): 185.0 horas
  ('Nyx', 'painted'): 128.0 horas
  ('Olmo', 'flat'): 18.0 horas
  ('Sela', 'flat'): 42.0 horas
  ('Gorr', 'painted'): 145.0 horas

Horas para ('Lyra', 'cel_shaded'): 142.0


**Pregunta Lógica 8.** Con base en la salida de la celda anterior:

a) ¿Cuál es el tiempo de producción registrado para el personaje Lyra con estilo cel_shaded?
b) El código siguiente intenta crear un diccionario empleando una lista como clave. ¿Tendrá éxito? Explica por qué razón, sin ejecutar la celda.

```python
invalid_lookup = {["Lyra", "cel_shaded"]: 142.0}
```

**Solución / Respuestas:**
- **a)** `142.0` horas.
- **b)** No tendrá éxito. En Python, las claves de diccionario deben ser objetos inmutables y con soporte para cálculo de hash (hashables). Las listas son mutables y por ende carecen de hash, por lo que la instrucción arrojará una excepción `TypeError: unhashable type: 'list'`. Para claves compuestas se deben emplear tuplas.

---
## Parte 3: Diccionarios -- Corrección del Dataset

### Ejercicio 3.1 -- Creación de una Copia de Trabajo Profunda (Deep Copy)

**Referencia.** Antes de modificar colecciones de datos, es una práctica estándar preservar la fuente original. Una copia superficial (shallow copy mediante `.copy()` o `[:]`) no duplica los diccionarios anidados; alterar un registro interno afectaría simultáneamente al original. Una copia profunda (`import copy; copy.deepcopy(objeto)`) clona recursivamente todas las estructuras internas.

Todas las correcciones a partir del Ejercicio 3.2 deben aplicarse sobre `characters` (la copia profunda). Ejecuta la celda siguiente antes de continuar.

In [18]:
# Crear una copia profunda del dataset -- ejecutar esta celda, no modificarla
import copy
characters = copy.deepcopy(characters_raw)
print(f"Copia de trabajo creada: {len(characters)} personajes.")
print("Modificar 'characters' no afectará a 'characters_raw'.")

Copia de trabajo creada: 20 personajes.
Modificar 'characters' no afectará a 'characters_raw'.


### Ejercicio 3.2 -- Corrección de Valores Ausentes en el Rol

**Referencia.** Una clave específica dentro de un diccionario se actualiza mediante asignación directa: `d["clave"] = nuevo_valor`. Al manipular una lista de diccionarios, el registro de interés se localiza iterando y evaluando un campo unívoco como `name`. Un diccionario de correcciones asocia cada nombre a su valor corregido, evitando estructuras condicionales repetitivas.

**Ejemplo.** La siguiente celda aplica un mapeo de correcciones a una lista de registros.

In [19]:
# Ejemplo -- no modificar
sample = [
    {"name": "Alpha", "role": None},
    {"name": "Beta",  "role": "support"},
    {"name": "Gamma", "role": None},
]
corrections = {"Alpha": "protagonist", "Gamma": "antagonist"}
for record in sample:
    if record["name"] in corrections:
        record["role"] = corrections[record["name"]]
print("Tras la corrección:", [(r["name"], r["role"]) for r in sample])

Tras la corrección: [('Alpha', 'protagonist'), ('Beta', 'support'), ('Gamma', 'antagonist')]


**Tarea.** En la lista `characters`, aplica las siguientes correcciones en el atributo `role`:
- Asigna `"support"` al personaje con nombre `"Crux"`.
- Asigna `"antagonist"` al personaje con nombre `"Rex"`.

Tras aplicar las correcciones, imprime los personajes modificados y luego el conteo de personajes por cada rol para validar los resultados.

In [20]:
# Solución 3.2
role_corrections = {
    "Crux": "support",
    "Rex":  "antagonist",
}

# Aplicar correcciones a la copia de trabajo
for c in characters:
    if c["name"] in role_corrections:
        c["role"] = role_corrections[c["name"]]

# Verificar: imprimir los personajes corregidos
for c in characters:
    if c["name"] in role_corrections:
        print(f"Corregido: {c['name']} -> role: {c['role']}")

print()
# Contar personajes por rol
for role in ["protagonist", "antagonist", "support"]:
    count = len([c for c in characters if c["role"] == role])
    print(f"{role}: {count}")

Corregido: Crux -> role: support
Corregido: Rex -> role: antagonist

protagonist: 4
antagonist: 7
support: 9


**Pregunta Lógica 9.** Con base en la salida de la celda anterior:

a) Tras la corrección, ¿cuántos personajes tienen asignado el rol `"antagonist"`?
b) Tras la corrección, ¿cuántos personajes tienen asignado el rol `"support"`?
c) ¿La sumatoria de los tres roles suma exactamente 20? De no ser así, ¿qué motivo explica la diferencia?

**Solución / Respuestas:**
- **a)** `7` antagonistas (`Kael`, `Zorn`, `Drak`, `Thorn`, `Rex`, `Nyx`, `Gorr`).
- **b)** `9` personajes de apoyo/soporte (`Mira`, `Fenn`, `Crux`, `Vela`, `Pip`, `Lune`, `Cleo`, `Olmo`, `Sela`).
- **c)** Sí, la suma es exacta: 4 (protagonistas) + 7 (antagonistas) + 9 (soporte) = `20`. Todos los 20 personajes poseen ahora un rol válido definido, sin valores nulos.

### Ejercicio 3.3 -- Corrección del Tipo de Dato de fps

**Referencia.** La función integrada `int(valor)` convierte una cadena que representa un número en un entero auténtico. La comprobación `isinstance(valor, str)` permite identificar selectivamente los valores almacenados como texto sin necesidad de especificar nombres de personajes de manera fija.

**Ejemplo.** La siguiente celda identifica y convierte valores de fps guardados como cadenas de texto.

In [21]:
# Ejemplo -- no modificar
sample = [
    {"name": "A", "fps": 24},
    {"name": "B", "fps": "12"},
    {"name": "C", "fps": 8},
]
for r in sample:
    if isinstance(r["fps"], str):
        r["fps"] = int(r["fps"])
print("Tras la corrección:")
for r in sample:
    print(f"  {r['name']}: fps={r['fps']}  tipo={type(r['fps']).__name__}")

Tras la corrección:
  A: fps=24  tipo=int
  B: fps=12  tipo=int
  C: fps=8  tipo=int


**Tarea.** Itera sobre `characters` y convierte todo valor de `fps` almacenado como cadena a tipo entero. Al finalizar, imprime el nombre, valor de fps y tipo de dato de cada personaje para certificar la consistencia.

In [22]:
# Solución 3.3
# Corregir el tipo de fps donde sea necesario
for c in characters:
    if isinstance(c["fps"], str):
        c["fps"] = int(c["fps"])

# Verificar: imprimir valor de fps y tipo para cada personaje
for c in characters:
    print(f"  {c['name']:10s}  fps={c['fps']:3}  tipo={type(c['fps']).__name__}")

  Lyra        fps= 24  tipo=int
  Kael        fps= 24  tipo=int
  Mira        fps= 12  tipo=int
  Zorn        fps= 24  tipo=int
  Fenn        fps= 12  tipo=int
  Seraph      fps= 24  tipo=int
  Crux        fps= 12  tipo=int
  Vela        fps= 12  tipo=int
  Drak        fps= 24  tipo=int
  Pip         fps=  8  tipo=int
  Asha        fps= 24  tipo=int
  Thorn       fps= 12  tipo=int
  Lune        fps= 12  tipo=int
  Rex         fps= 24  tipo=int
  Cleo        fps= 12  tipo=int
  Blaze       fps= 24  tipo=int
  Nyx         fps= 24  tipo=int
  Olmo        fps=  8  tipo=int
  Sela        fps= 12  tipo=int
  Gorr        fps= 24  tipo=int


**Pregunta Lógica 10.** Basándote en la salida generada por la celda anterior:

a) ¿Cuál es el valor de fps y su tipo de dato para el personaje Kael tras la corrección?
b) ¿Cuál es el valor de fps y su tipo de dato para el personaje Thorn tras la corrección?
c) ¿Cuántos valores distintos de fps se observan entre los 20 personajes? Enuméralos.

**Solución / Respuestas:**
- **a)** `fps = 24`, de tipo `int`.
- **b)** `fps = 12`, de tipo `int`.
- **c)** `3` valores distintos de fps: `8`, `12` y `24` fotogramas por segundo (conjunto `{8, 12, 24}`).

### Ejercicio 3.4 -- Corrección de Valores Ausentes en has_special_fx

**Referencia.** El atributo `has_special_fx` es un indicador binario: `1` si el personaje emplea efectos visuales especiales (partículas, resplandores), `0` en caso contrario. Cuando el dato figura como `None`, el estándar prudente en planificación de producción consiste en fijar el valor por defecto en `0`.

**Tarea.** Itera sobre `characters` y asigna `0` a todo campo `has_special_fx` que sea `None`. Imprime el nombre y el valor corregido de cada personaje afectado. Posteriormente, imprime la cantidad de personajes con y sin efectos especiales.

In [23]:
# Solución 3.4
# Corregir valores faltantes en has_special_fx
for c in characters:
    if c["has_special_fx"] is None:
        c["has_special_fx"] = 0
        print(f"Corregido: {c['name']} -> has_special_fx = {c['has_special_fx']}")

# Contar personajes con y sin efectos especiales
count_fx    = len([c for c in characters if c["has_special_fx"] == 1])
count_no_fx = len([c for c in characters if c["has_special_fx"] == 0])
print(f"\nPersonajes con efectos especiales:    {count_fx}")
print(f"Personajes sin efectos especiales:    {count_no_fx}")

Corregido: Zorn -> has_special_fx = 0
Corregido: Rex -> has_special_fx = 0

Personajes con efectos especiales:    10
Personajes sin efectos especiales:    10


**Pregunta Lógica 11.** Basándote en la salida de la celda anterior:

a) ¿Qué personajes fueron corregidos en su campo `has_special_fx` pasando de `None` a `0`?
b) Tras la corrección, ¿cuántos personajes tienen `has_special_fx` igual a `1`?
c) Tras la corrección, ¿cuántos personajes tienen `has_special_fx` igual a `0`?

**Solución / Respuestas:**
- **a)** `Zorn` y `Rex`.
- **b)** `10` personajes.
- **c)** `10` personajes.

### Ejercicio 3.5 -- Completar Paletas Incompletas

**Referencia.** Una lista asociada al valor de una clave en un diccionario puede expandirse mediante el método `.append(elemento)`, el cual adiciona un elemento modificando la lista in situ. Cada color debe agregarse en formato de tupla `(R, G, B)`.

**Ejemplo.** La siguiente celda añade colores a una paleta alojada en un diccionario.

In [24]:
# Ejemplo -- no modificar
char_sample = {"name": "Demo", "palette": [(100, 100, 100)]}
print("Antes:", char_sample["palette"])

char_sample["palette"].append((200, 200, 200))
char_sample["palette"].append((50, 50, 50))
print("Después:", char_sample["palette"])
print("Colores:", len(char_sample["palette"]))

Antes: [(100, 100, 100)]
Después: [(100, 100, 100), (200, 200, 200), (50, 50, 50)]
Colores: 3


**Tarea.** Agrega los siguientes colores a las paletas incompletas en `characters`. Tras actualizar cada registro, imprime el nombre del personaje y su paleta completa actualizada.

| Personaje | Colores a incorporar |
|-----------|----------------------|
| Mira      | `(240, 250, 255)` |
| Drak      | `(255, 50, 50)` y `(80, 0, 0)` |
| Olmo      | `(80, 140, 60)` y `(240, 255, 220)` |

In [25]:
# Solución 3.5
palette_additions = {
    "Mira": [(240, 250, 255)],
    "Drak": [(255, 50, 50), (80, 0, 0)],
    "Olmo": [(80, 140, 60), (240, 255, 220)],
}

for c in characters:
    if c["name"] in palette_additions:
        for color in palette_additions[c["name"]]:
            c["palette"].append(color)
        print(f"{c['name']}: {len(c['palette'])} colores -> {c['palette']}")

Mira: 3 colores -> [(200, 230, 255), (120, 180, 240), (240, 250, 255)]
Drak: 4 colores -> [(20, 20, 20), (150, 0, 0), (255, 50, 50), (80, 0, 0)]
Olmo: 3 colores -> [(160, 200, 120), (80, 140, 60), (240, 255, 220)]


**Pregunta Lógica 12.** Basándote en la salida generada por la celda anterior:

a) ¿Cuántos colores contiene la paleta de Mira tras la corrección?
b) ¿Cuántos colores contiene la paleta de Olmo tras la corrección?
c) Escribe una comprensión de listas que verifique que ningún personaje tenga menos de 3 colores en su paleta, retornando una lista vacía si todas las paletas son válidas.

**Solución / Respuestas:**
- **a)** `3` colores.
- **b)** `3` colores.
- **c)** `[c["name"] for c in characters if len(c["palette"]) < 3]` (devuelve `[]` confirmando que todas las paletas tienen al menos 3 colores).

### Ejercicio 3.6 -- Comprensión de Diccionarios: Conteo de Estados por Personaje

**Referencia.** Una comprensión de diccionarios genera un nuevo diccionario a partir de un iterable en una única expresión: `{clave_expr: valor_expr for item in iterable}`. Permite además aplicar condiciones de filtrado.

**Ejemplo.** La siguiente celda asocia cada título de juego a la cantidad de caracteres que componen su nombre.

In [26]:
# Ejemplo -- no modificar
game_list = ["Celeste", "Hades", "Cuphead", "Shovel Knight"]
name_lengths = {title: len(title) for title in game_list}
print("Longitudes de nombres:", name_lengths)

longest = max(name_lengths, key=lambda title: name_lengths[title])
print("Título más largo:", longest)

Longitudes de nombres: {'Celeste': 7, 'Hades': 5, 'Cuphead': 7, 'Shovel Knight': 13}
Título más largo: Shovel Knight


**Tarea.** Utilizando la lista depurada `characters`, construye un diccionario llamado `state_counts` donde cada clave sea el nombre del personaje y cada valor represente la cantidad de estados de animación (`len(states)`). Imprime el diccionario y determina cuál es el personaje con el mayor número de estados.

In [27]:
# Solución 3.6
# Construir el diccionario
state_counts = {c["name"]: len(c["states"]) for c in characters}

# Imprimir todas las entradas
print("Estados de animación por personaje:")
for name, count in state_counts.items():
    print(f"  {name}: {count}")

# Identificar el personaje con mayor cantidad de estados
most_complex = max(state_counts, key=lambda name: state_counts[name])
print(f"\nPersonaje más complejo: {most_complex} ({state_counts[most_complex]} estados)")

Estados de animación por personaje:
  Lyra: 5
  Kael: 5
  Mira: 2
  Zorn: 4
  Fenn: 3
  Seraph: 6
  Crux: 3
  Vela: 4
  Drak: 5
  Pip: 2
  Asha: 5
  Thorn: 4
  Lune: 3
  Rex: 4
  Cleo: 4
  Blaze: 7
  Nyx: 4
  Olmo: 1
  Sela: 3
  Gorr: 6

Personaje más complejo: Blaze (7 estados)


**Pregunta Lógica 13.** Basándote en la salida producida por la celda anterior:

a) ¿Qué personaje posee la mayor cantidad de estados de animación?
b) ¿Cuántos estados posee dicho personaje? Enumera cada estado por su nombre.
c) ¿Cuántos personajes poseen exactamente 3 estados de animación?

**Solución / Respuestas:**
- **a)** `Blaze`.
- **b)** `7` estados: `"idle"`, `"walk"`, `"run"`, `"attack"`, `"special"`, `"fall"` y `"victory"`.
- **c)** `4` personajes tienen exactamente 3 estados (`Crux`, `Fenn`, `Lune` y `Sela`).

---
## Parte 4: Integración -- Construcción del Dataset Limpio Final

Todas las inconsistencias de calidad de datos han sido resueltas. En esta etapa final, el estudiante debe calcular campos derivados e incorporarlos a cada registro de `characters`, obteniendo el dataset definitivo que será consumido en la Parte 2 (análisis exploratorio con Pandas) y en la Parte 3 (regresión lineal).

A continuación se detallan los campos calculados a incorporar en cada diccionario de personaje:

| Nuevo Campo          | Fórmula |
|----------------------|---------|
| `num_colors`         | `len(palette)` |
| `avg_brightness`     | Media de `(R+G+B)/3` de todos los colores de la paleta, redondeada a 2 decimales |
| `color_contrast`     | Brillo máximo de color menos brillo mínimo de color, redondeado a 2 decimales |
| `num_states`         | `len(states)` |
| `num_frames_total`   | `num_states * fps` |

### Ejercicio 4.1 -- Incorporación de Campos Calculados a Cada Personaje

**Referencia.** Un nuevo par clave-valor se agrega a un diccionario preexistente mediante asignación: `d["nueva_clave"] = valor`. Al iterar sobre una lista de diccionarios, cada elemento es una referencia al diccionario original, por lo que las modificaciones se aplican directamente in situ.

In [28]:
# Solución 4.1
for c in characters:
    palette = c["palette"]

    # num_colors: cantidad total de colores en la paleta
    c["num_colors"] = len(palette)

    # Calcular el brillo de cada color mediante desempaquetado de tuplas
    brightness_values = [(r + g + b) / 3 for r, g, b in palette]

    # avg_brightness: brillo medio de todos los colores de la paleta
    c["avg_brightness"] = round(sum(brightness_values) / len(brightness_values), 2)

    # color_contrast: diferencia entre el color más luminoso y el más oscuro
    c["color_contrast"] = round(max(brightness_values) - min(brightness_values), 2)

    # num_states: cantidad de estados de animación definidos para este personaje
    c["num_states"] = len(c["states"])

    # num_frames_total: estimación total de fotogramas clave
    c["num_frames_total"] = c["num_states"] * c["fps"]

print("Campos derivados añadidos exitosamente.")
print("Lista de campos presentes en cada registro:", list(characters[0].keys()))

Campos derivados añadidos exitosamente.
Lista de campos presentes en cada registro: ['name', 'role', 'palette', 'states', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'production_hours', 'num_colors', 'avg_brightness', 'color_contrast', 'num_states', 'num_frames_total']


### Ejercicio 4.2 -- Verificación del Dataset Limpio

**Referencia.** Un paso formal de validación confirma que todos los registros se encuentran completos y consistentes antes de transferir los datos a la siguiente fase. La siguiente celda verifica que no existan valores `None` en campos requeridos, valida que `fps` sea de tipo `int`, comprueba que las paletas cuenten con al menos 3 colores y constata la existencia de todos los campos derivados calculados.

Ejecuta esta celda tras finalizar el Ejercicio 4.1. Si se reporta alguna observación, revisa el ejercicio correspondiente antes de continuar.

In [29]:
# Celda de verificación -- ejecutar tras completar el Ejercicio 4.1

required_fields = [
    "name", "role", "palette", "states", "fps", "has_special_fx",
    "style", "screen_time_ratio", "production_hours",
    "num_colors", "avg_brightness", "color_contrast",
    "num_states", "num_frames_total"
]

issues = []

for c in characters:
    for field in required_fields:
        if c.get(field) is None:
            issues.append(f"{c['name']}: el campo '{field}' es None o no existe")
    if not isinstance(c.get("fps"), int):
        issues.append(f"{c['name']}: fps no es entero (tipo actual: {type(c.get('fps')).__name__})")
    if len(c.get("palette", [])) < 3:
        issues.append(f"{c['name']}: la paleta contiene {len(c.get('palette',[]))} color(es) -- el mínimo requerido es 3")

if issues:
    print(f"Verificación fallida. Se detectaron {len(issues)} problema(s):")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("Verificación exitosa: los 20 registros están completos y validados.")
    print(f"Total de campos por registro: {len(required_fields)}")

Verificación exitosa: los 20 registros están completos y validados.
Total de campos por registro: 14


### Ejercicio 4.3 -- Estadísticas Resumen del Dataset Depurado

**Referencia.** Las funciones integradas de Python `min()`, `max()` y `sum()` procesan cualquier objeto iterable. Combinadas con comprensiones de listas y funciones `lambda`, permiten extraer estadísticas descriptivas fundamentales sin depender de librerías externas.

**Tarea.** Calcula e imprime las siguientes métricas a partir del dataset depurado `characters`:
- El promedio de `production_hours` entre los 20 personajes, redondeado a 2 decimales.
- El nombre y valor del personaje con el mayor `num_frames_total`.
- El nombre y valor del personaje con el mayor `screen_time_ratio`.
- El nombre y valor del personaje con la mayor cantidad de `production_hours`.

In [30]:
# Solución 4.3
# Horas de producción promedio entre todos los personajes
all_hours = [c["production_hours"] for c in characters]
avg_hours = sum(all_hours) / len(all_hours)
print("Horas de producción promedio:", round(avg_hours, 2))

# Personaje con mayor cantidad total de fotogramas
top_frames = max(characters, key=lambda c: c["num_frames_total"])
print(f"Mayor num_frames_total: {top_frames['name']} ({top_frames['num_frames_total']} fotogramas)")

# Personaje con mayor ratio de tiempo en pantalla
top_screen = max(characters, key=lambda c: c["screen_time_ratio"])
print(f"Mayor screen_time_ratio: {top_screen['name']} ({top_screen['screen_time_ratio']})")

# Personaje con mayor tiempo de producción
top_hours = max(characters, key=lambda c: c["production_hours"])
print(f"Mayor production_hours: {top_hours['name']} ({top_hours['production_hours']} horas)")

Horas de producción promedio: 96.33
Mayor num_frames_total: Blaze (168 fotogramas)
Mayor screen_time_ratio: Lyra (0.38)
Mayor production_hours: Blaze (185.0 horas)


**Pregunta Lógica 14.** Basándote en la salida generada por la celda anterior:

a) ¿Cuál es el promedio de horas de producción entre los 20 personajes?
b) ¿Qué personaje requirió la mayor cantidad de horas de producción y cuál es dicho valor?
c) ¿El personaje con mayor `num_frames_total` coincide con el personaje de mayor `production_hours`? Si discrepan, ¿qué evidencia aporta esto respecto a si la cantidad de fotogramas determina de forma aislada el costo de producción?

**Solución / Respuestas:**
- **a)** `96.33` horas (sumatoria: `1926.5` / 20 = 96.325 -> `96.33`).
- **b)** `Blaze` con `185.0` horas de producción.
- **c)** Sí, en este conjunto de datos `Blaze` reúne tanto el mayor total de fotogramas (168 fotogramas) como el mayor costo en horas (185.0 horas). No obstante, el contraste entre los restantes personajes demuestra que la cantidad de fotogramas no actúa de forma univariable: por ejemplo, `Seraph` y `Gorr` poseen exactamente 144 fotogramas cada uno, pero `Seraph` demandó 168.0 horas frente a las 145.0 horas de `Gorr` debido al estilo de renderizado pintado (`painted`), la proporción de pantalla y la presencia de efectos especiales.

---
## Paso Final -- Exportación del Dataset Limpio

La siguiente celda exporta el conjunto de datos validado a un archivo CSV denominado `characters_clean.csv` en el directorio de trabajo. Este archivo constituirá la entrada de datos para la Parte 2. La exportación emplea exclusivamente módulos estándar de Python, sin requerir librerías externas.

In [31]:
# Exportar el dataset limpio a CSV -- ejecutar tras superar la verificación
import csv

export_fields = [
    "name", "role", "fps", "has_special_fx", "style",
    "screen_time_ratio", "num_colors", "avg_brightness", "color_contrast",
    "num_states", "num_frames_total", "production_hours"
]

with open("characters_clean.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=export_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(characters)

print("Exportación completada: characters_clean.csv")
print(f"Registros exportados: {len(characters)}")
print(f"Campos exportados: {export_fields}")

Exportación completada: characters_clean.csv
Registros exportados: 20
Campos exportados: ['name', 'role', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'num_colors', 'avg_brightness', 'color_contrast', 'num_states', 'num_frames_total', 'production_hours']


---
## Guía Rápida de Referencia: Operaciones de Estructuras de Datos Utilizadas

| Operación | Sintaxis |
|-----------|----------|
| Acceso por índice en lista | `lista[0]`, `lista[-1]`, `lista[i]` |
| Longitud de lista | `len(lista)` |
| Comprensión de listas | `[expr for x in lista]` |
| Comprensión con filtro | `[expr for x in lista if condicion]` |
| Comprobación de valor nulo | `valor is None` |
| Comprobación de tipo | `isinstance(valor, tipo)` |
| Conversión de tipo | `int(valor)`, `float(valor)` |
| Control de excepciones | `try: ... except TypeError as e: ...` |
| Acceso a elementos de tupla | `t[0]`, `t[1]`, `t[-1]` |
| Desempaquetado de tuplas | `r, g, b = tupla_color` |
| Iteración con enumerate | `for i, (a, b) in enumerate(lista):` |
| Acceso a clave en diccionario | `d["clave"]` |
| Acceso seguro a diccionario | `d.get("clave", default)` |
| Actualización de diccionario | `d["clave"] = nuevo_valor` |
| Comprensión de diccionarios | `{k_expr: v_expr for item in iterable}` |
| Inserción en lista | `lista.append(elemento)` |
| Búsqueda de máximo por campo | `max(lista, key=lambda x: x["campo"])` |
| Clonación profunda | `import copy; copy.deepcopy(objeto)` |
| Escritura en archivo CSV | `csv.DictWriter(f, fieldnames=...).writerows(datos)` |